# Formal LSTM Cache Action Predictor

**Goal:** train a real LSTM model for memory/cache behavior, not a toy smoke test.

This notebook implements the design you described:

```text
PC hash / instruction semantic / address / delta history
hit-miss history / time phase / cache pressure / optional SPP signals
        ↓
Embedding + LSTM memory
        ↓
multi-task cache-action heads
        ├── next useful delta class
        ├── future hit tendency
        ├── cache bypass / low-priority insertion decision
        └── timing / reuse-distance bucket
        ↓
export action table for real ChampSim replay
```

Important framing:

```text
This is NOT only an SPP filter.

SPP/candidate features may be inputs, but the NN's job is broader:
under what pattern, at what time/cache state, what memory/cache action is useful?
```


## A. Colab / GitHub pull

Run this at the beginning in Colab. If you already ran your PAT clone cell, this cell only resets to the latest `origin/main`.

This notebook assumes the repo root is `/content/cache_arch` on Colab, or the current working tree on cluster.


In [ ]:
from pathlib import Path
import os, subprocess

REPO_ROOT = Path('/content/cache_arch')
if REPO_ROOT.exists():
    os.chdir(REPO_ROOT)
    subprocess.run(['git', 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', 'reset', '--hard', 'origin/main'], check=True)
else:
    # For private repo clone, run your existing getpass/PAT clone cell first.
    raise FileNotFoundError(
        "Repo not found at /content/cache_arch. "
        "Run your GitHub PAT clone cell first, then rerun this cell."
    )

print('cwd =', Path.cwd())
subprocess.run(['git', 'log', '--oneline', '-3'], check=True)
subprocess.run(['git', 'status', '--short'], check=True)


## 0. What this notebook expects

Put real ChampSim-derived event tables under one of these paths:

```text
formal_NN_training/data/*.csv
formal_NN_training/data/*.parquet
formal_NN_training/data/generated/*.csv
formal_NN_training/data/generated/*.parquet
```

Minimum required columns:

```text
pc, addr
```

Better columns:

```text
trace, event_id, cycle, pc, addr, hit, is_store,
spp_delta, spp_conf,
mshr_occupancy, l2_occupancy, bandwidth_pressure,
semantic_class
```

No synthetic fallback is provided. If real data is missing, the notebook intentionally stops.


In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

from pathlib import Path
import os, json, math, glob, time, hashlib, warnings
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# Auto-detect whether the notebook is launched from repo root or from formal_NN_training/.
_cwd = Path.cwd()
if _cwd.name == "formal_NN_training":
    PROJECT_ROOT = _cwd
elif (_cwd / "formal_NN_training").exists():
    PROJECT_ROOT = _cwd / "formal_NN_training"
else:
    PROJECT_ROOT = _cwd

CONFIG = {
    # Real data only. This notebook intentionally does not create toy/smoke data.
    "data_globs": [
        "data/*.parquet",
        "data/*.csv",
        "data/generated/*.parquet",
        "data/generated/*.csv",
        "formal_NN_training/data/*.parquet",
        "formal_NN_training/data/*.csv",
        "formal_NN_training/data/generated/*.parquet",
        "formal_NN_training/data/generated/*.csv",
        "../formal_NN_training/data/*.parquet",
        "../formal_NN_training/data/*.csv",
        "../formal_NN_training/data/generated/*.parquet",
        "../formal_NN_training/data/generated/*.csv",
    ],

    # Memory/cache assumptions.
    "cache_line_bytes": 64,
    "page_bytes": 4096,

    # Sequence modeling.
    "seq_len": 64,
    "prediction_horizon": 1,

    # Vocabularies. Increasing these may improve accuracy but costs memory.
    "pc_hash_buckets": 8192,
    "top_delta_vocab": 512,
    "offset_vocab": 64,
    "semantic_hash_buckets": 64,

    # Reuse/bypass label construction.
    # Bypass label = 1 when the line is not reused soon.
    "bypass_reuse_distance_events": 2048,
    "no_reuse_distance_cap": 1_000_000,

    # Timing bins measured in cycles if cycle column exists; otherwise event distance.
    "time_bin_edges": [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 4096, 16384, 65536, 262144, 1048576],

    # Split: use early part of each trace for training and later part for validation.
    # This is intentionally stricter than random split.
    "train_fraction_per_trace": 0.80,

    # Model.
    "emb_dim": 48,
    "cont_dim": 8,
    "hidden_dim": 192,
    "num_layers": 2,
    "dropout": 0.15,

    # Training.
    "batch_size": 256,
    "epochs": 20,
    "lr": 2e-3,
    "weight_decay": 1e-5,
    "grad_clip": 1.0,
    "num_workers": 2,
    "early_stop_patience": 4,

    # Loss weights.
    "loss_delta": 1.00,
    "loss_hit": 0.25,
    "loss_bypass": 0.75,
    "loss_timing": 0.35,

    # Accuracy goal gates. These are sanity gates, not guaranteed.
    # Structured microbenchmarks may exceed these. Irregular SPEC/GAP traces may not.
    "target_delta_top1": 0.90,
    "target_delta_top5": 0.97,
    "target_bypass_f1": 0.90,

    # Export.
    "artifact_dir": "artifacts",
    "export_predictions": True,
    "prediction_threshold_prefetch": 0.50,
    "prediction_threshold_bypass": 0.60,
}

ARTIFACT_DIR = PROJECT_ROOT / CONFIG["artifact_dir"]
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("DEVICE:", DEVICE)


## B. Optional: generate real SPP trace data

Run this only on a machine where `external/ChampSim/` and `traces/` exist.

It uses the GitHub script:

```text
formal_NN_training/scripts/01_run_spp_trace_dump.sh
```

Output consumed by the training cells below:

```text
formal_NN_training/data/generated/lstm_events_<TRACE>.csv
```


In [ ]:
import os, subprocess
os.environ.setdefault('TRACE', '602.gcc_s-734B')
os.environ.setdefault('WARMUP', '25000000')
os.environ.setdefault('SIM', '25000000')
os.environ.setdefault('BUILD', '1')
os.environ.setdefault('PATCH_SPP', '1')

subprocess.run(['bash', 'formal_NN_training/scripts/01_run_spp_trace_dump.sh'], check=True)


In [ ]:
# ============================================================
# 2. Utility functions
# ============================================================

def parse_int_maybe_hex(x):
    """Parse integer or hex string. Keeps NaN as NaN."""
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, np.integer)):
        return int(x)
    if isinstance(x, float):
        return int(x)
    s = str(x).strip()
    if s.startswith(("0x", "0X")):
        return int(s, 16)
    # Some logs use decimal strings.
    return int(float(s))

def stable_hash_int(x, mod: int) -> int:
    """Stable hash for PCs/semantic strings. Python hash is randomized, so use blake2b."""
    if pd.isna(x):
        return 0
    b = str(x).encode("utf-8")
    h = hashlib.blake2b(b, digest_size=8).digest()
    return int.from_bytes(h, "little") % mod

def bucketize(values: np.ndarray, edges: List[int]) -> np.ndarray:
    """Return bin ids in [0, len(edges)]."""
    return np.searchsorted(np.asarray(edges, dtype=np.float64), values, side="right").astype(np.int64)

def safe_float_col(df, name, default=0.0):
    if name not in df.columns:
        return np.full(len(df), default, dtype=np.float32)
    return pd.to_numeric(df[name], errors="coerce").fillna(default).astype(np.float32).to_numpy()

def find_real_event_files() -> List[Path]:
    files = []
    for pat in CONFIG["data_globs"]:
        files.extend(Path(PROJECT_ROOT).glob(pat))
    # Deduplicate while preserving order.
    out = []
    seen = set()
    for p in files:
        rp = p.resolve()
        if rp not in seen:
            out.append(p)
            seen.add(rp)
    return out

event_files = find_real_event_files()
print("Found event files:")
for f in event_files[:20]:
    print("  ", f)
if not event_files:
    raise FileNotFoundError(
        "No real event table found. Put ChampSim-derived CSV/Parquet files under "
        "formal_NN_training/data/ or formal_NN_training/data/generated/. "
        "This notebook intentionally does not create toy/smoke data."
    )


In [ ]:
# ============================================================
# 3. Load real event table
# ============================================================

def load_one_event_file(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")
    df["source_file"] = str(path)
    return df

dfs = []
for path in event_files:
    df_i = load_one_event_file(path)
    dfs.append(df_i)
    print(f"loaded {path}: {len(df_i):,} rows, columns={list(df_i.columns)[:12]}")

df = pd.concat(dfs, ignore_index=True)
print("Total rows:", f"{len(df):,}")
print("Columns:", list(df.columns))

required = {"pc", "addr"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}. Minimum schema is pc, addr.")

# Normalize trace id.
if "trace" not in df.columns:
    df["trace"] = df["source_file"].astype(str)

# Normalize event id.
if "event_id" not in df.columns:
    df["event_id"] = np.arange(len(df), dtype=np.int64)

# Parse PC/addr if they are strings.
if not np.issubdtype(df["pc"].dtype, np.integer):
    df["pc_int"] = df["pc"].map(parse_int_maybe_hex).astype(np.int64)
else:
    df["pc_int"] = df["pc"].astype(np.int64)

if not np.issubdtype(df["addr"].dtype, np.integer):
    df["addr_int"] = df["addr"].map(parse_int_maybe_hex).astype(np.int64)
else:
    df["addr_int"] = df["addr"].astype(np.int64)

# Sort by trace then time/order.
sort_cols = ["trace"]
if "cycle" in df.columns:
    sort_cols.append("cycle")
sort_cols.append("event_id")
df = df.sort_values(sort_cols).reset_index(drop=True)

df.head()


In [ ]:
# ============================================================
# 4. Derive cache/memory features and labels
# ============================================================

CL = CONFIG["cache_line_bytes"]
PAGE = CONFIG["page_bytes"]

df["line_addr"] = (df["addr_int"] // CL).astype(np.int64)
df["page_id"] = (df["addr_int"] // PAGE).astype(np.int64)
df["line_offset_in_page"] = (df["line_addr"] % (PAGE // CL)).astype(np.int64)

# Delta in cache lines within each trace.
df["prev_line_addr"] = df.groupby("trace")["line_addr"].shift(1)
df["delta"] = (df["line_addr"] - df["prev_line_addr"]).fillna(0).astype(np.int64)

# Next delta label.
H = CONFIG["prediction_horizon"]
df["future_line_addr"] = df.groupby("trace")["line_addr"].shift(-H)
df["future_delta"] = (df["future_line_addr"] - df["line_addr"]).fillna(0).astype(np.int64)

# Hit/miss feature and label.
# If no hit column exists, current hit feature is unknown=2 and future hit label is ignored via mask.
if "hit" in df.columns:
    df["hit_int"] = pd.to_numeric(df["hit"], errors="coerce").fillna(0).astype(np.int64).clip(0, 1)
    df["future_hit"] = df.groupby("trace")["hit_int"].shift(-H).fillna(0).astype(np.int64).clip(0, 1)
    df["has_hit_label"] = 1
else:
    df["hit_int"] = 2  # unknown token
    df["future_hit"] = 0
    df["has_hit_label"] = 0

# Access type.
if "is_store" in df.columns:
    df["access_type"] = pd.to_numeric(df["is_store"], errors="coerce").fillna(0).astype(np.int64).clip(0, 1)
elif "type" in df.columns:
    df["access_type"] = df["type"].astype(str).str.lower().str.contains("store|write").astype(np.int64)
else:
    df["access_type"] = 0

# PC hash and semantic hash.
df["pc_id"] = df["pc_int"].map(lambda x: stable_hash_int(x, CONFIG["pc_hash_buckets"])).astype(np.int64)

if "semantic_class" in df.columns:
    df["semantic_id"] = df["semantic_class"].map(lambda x: stable_hash_int(x, CONFIG["semantic_hash_buckets"])).astype(np.int64)
elif "opcode" in df.columns:
    df["semantic_id"] = df["opcode"].map(lambda x: stable_hash_int(x, CONFIG["semantic_hash_buckets"])).astype(np.int64)
else:
    df["semantic_id"] = 0

# Time delta feature.
if "cycle" in df.columns:
    df["cycle_num"] = pd.to_numeric(df["cycle"], errors="coerce").fillna(method="ffill").fillna(0).astype(np.int64)
    df["prev_cycle"] = df.groupby("trace")["cycle_num"].shift(1).fillna(0).astype(np.int64)
    df["dt"] = (df["cycle_num"] - df["prev_cycle"]).clip(lower=0).astype(np.int64)
else:
    df["cycle_num"] = np.arange(len(df), dtype=np.int64)
    df["dt"] = 1

df["dt_bucket"] = bucketize(df["dt"].to_numpy(), CONFIG["time_bin_edges"])

# Reuse distance and bypass label.
# next_pos_same_line is computed per trace+line_addr.
df["pos_in_trace"] = df.groupby("trace").cumcount().astype(np.int64)
df["next_pos_same_line"] = df.groupby(["trace", "line_addr"])["pos_in_trace"].shift(-1)
reuse_dist = (df["next_pos_same_line"] - df["pos_in_trace"]).fillna(CONFIG["no_reuse_distance_cap"])
df["reuse_distance_events"] = reuse_dist.astype(np.int64)

# Bypass = likely dead or not reused soon.
df["bypass_label"] = (df["reuse_distance_events"] > CONFIG["bypass_reuse_distance_events"]).astype(np.int64)

# Timing / reuse bucket label.
df["timing_label"] = bucketize(
    df["reuse_distance_events"].clip(upper=CONFIG["no_reuse_distance_cap"]).to_numpy(),
    CONFIG["time_bin_edges"],
)

# Optional SPP / cache state continuous features.
# These are NOT used as a filter target. They are just optional context.
cont_cols = []
for name in ["spp_conf", "mshr_occupancy", "l2_occupancy", "bandwidth_pressure"]:
    arr = safe_float_col(df, name, default=0.0)
    col = f"{name}_float"
    df[col] = arr
    cont_cols.append(col)

# If SPP delta exists, include as a categorical-like continuous signal normalized later.
if "spp_delta" in df.columns:
    df["spp_delta_float"] = pd.to_numeric(df["spp_delta"], errors="coerce").fillna(0.0).astype(np.float32)
else:
    df["spp_delta_float"] = 0.0
cont_cols.append("spp_delta_float")

# Add recent hit/miss rolling features if hit is known.
# Shift by one to keep it online-safe.
if "hit" in df.columns:
    df["recent_hit_rate"] = (
        df.groupby("trace")["hit_int"]
          .transform(lambda s: s.shift(1).rolling(64, min_periods=1).mean())
          .fillna(0.5)
          .astype(np.float32)
    )
else:
    df["recent_hit_rate"] = 0.5
cont_cols.append("recent_hit_rate")

print("continuous features:", cont_cols)
df[["trace", "event_id", "pc_int", "addr_int", "delta", "future_delta", "hit_int", "future_hit", "bypass_label", "timing_label"]].head()


In [ ]:
# ============================================================
# 5. Build delta vocabulary
# ============================================================

TOPK = CONFIG["top_delta_vocab"]

delta_counts = df["delta"].value_counts()
future_delta_counts = df["future_delta"].value_counts()

# Use both input and output deltas so common future labels are retained.
combined_counts = delta_counts.add(future_delta_counts, fill_value=0).sort_values(ascending=False)
top_deltas = combined_counts.head(TOPK).index.astype(np.int64).tolist()

PAD_ID = 0
UNK_ID = 1
DELTA_OFFSET = 2
delta_to_id = {int(d): i + DELTA_OFFSET for i, d in enumerate(top_deltas)}
id_to_delta = {i + DELTA_OFFSET: int(d) for i, d in enumerate(top_deltas)}

def map_delta_to_id(arr):
    return np.asarray([delta_to_id.get(int(x), UNK_ID) for x in arr], dtype=np.int64)

df["delta_id"] = map_delta_to_id(df["delta"].to_numpy())
df["future_delta_id"] = map_delta_to_id(df["future_delta"].to_numpy())

num_delta_classes = TOPK + DELTA_OFFSET
num_time_classes = len(CONFIG["time_bin_edges"]) + 1

coverage = (df["future_delta_id"] != UNK_ID).mean()
print(f"future delta top-{TOPK} coverage: {coverage:.3%}")
print("top-20 deltas:", top_deltas[:20])

vocab = {
    "PAD_ID": PAD_ID,
    "UNK_ID": UNK_ID,
    "DELTA_OFFSET": DELTA_OFFSET,
    "top_deltas": top_deltas,
    "delta_to_id": {str(k): int(v) for k, v in delta_to_id.items()},
    "id_to_delta": {str(k): int(v) for k, v in id_to_delta.items()},
    "config": CONFIG,
}
with open(ARTIFACT_DIR / "delta_vocab.json", "w") as f:
    json.dump(vocab, f, indent=2)
print("saved", ARTIFACT_DIR / "delta_vocab.json")


In [ ]:
# ============================================================
# 6. Train/validation split by time within each trace
# ============================================================

# Valid sequence end positions must have enough history and a future label.
seq_len = CONFIG["seq_len"]
valid_end = np.ones(len(df), dtype=bool)

# Need seq_len history within same trace.
pos = df.groupby("trace").cumcount().to_numpy()
valid_end &= pos >= (seq_len - 1)

# Need future line/delta available. Last H rows of a trace are invalid for next-delta label.
trace_sizes = df.groupby("trace")["trace"].transform("size").to_numpy()
valid_end &= pos < (trace_sizes - CONFIG["prediction_horizon"])

# Time split per trace.
train_mask = np.zeros(len(df), dtype=bool)
val_mask = np.zeros(len(df), dtype=bool)
for trace, g in df.groupby("trace", sort=False):
    idx = g.index.to_numpy()
    cut = int(len(idx) * CONFIG["train_fraction_per_trace"])
    train_mask[idx[:cut]] = True
    val_mask[idx[cut:]] = True

train_end_positions = np.where(valid_end & train_mask)[0]
val_end_positions = np.where(valid_end & val_mask)[0]

if len(train_end_positions) == 0 or len(val_end_positions) == 0:
    raise ValueError("Not enough rows for train/val sequences. Reduce seq_len or provide more trace events.")

df["is_train_row"] = train_mask
print("train sequences:", len(train_end_positions))
print("val sequences:", len(val_end_positions))


In [ ]:
# ============================================================
# 7. Dataset
# ============================================================

feature_arrays = {
    "pc_id": df["pc_id"].to_numpy(np.int64),
    "delta_id": df["delta_id"].to_numpy(np.int64),
    "offset_id": df["line_offset_in_page"].to_numpy(np.int64),
    "hit_id": df["hit_int"].to_numpy(np.int64).clip(0, 2),
    "access_id": df["access_type"].to_numpy(np.int64).clip(0, 1),
    "semantic_id": df["semantic_id"].to_numpy(np.int64),
    "dt_bucket": df["dt_bucket"].to_numpy(np.int64),
    "cont": df[cont_cols].to_numpy(np.float32),
}

label_arrays = {
    "delta": df["future_delta_id"].to_numpy(np.int64),
    "hit": df["future_hit"].to_numpy(np.int64),
    "hit_mask": df["has_hit_label"].to_numpy(np.float32),
    "bypass": df["bypass_label"].to_numpy(np.int64),
    "timing": df["timing_label"].to_numpy(np.int64),
}

# Normalize continuous features using training rows only.
cont_train = feature_arrays["cont"][df["is_train_row"].to_numpy().astype(bool)]
cont_mean = cont_train.mean(axis=0)
cont_std = cont_train.std(axis=0) + 1e-6
feature_arrays["cont"] = ((feature_arrays["cont"] - cont_mean) / cont_std).astype(np.float32)

norm_stats = {
    "cont_cols": cont_cols,
    "mean": cont_mean.tolist(),
    "std": cont_std.tolist(),
}
with open(ARTIFACT_DIR / "continuous_feature_norm.json", "w") as f:
    json.dump(norm_stats, f, indent=2)

class CacheSequenceDataset(Dataset):
    def __init__(self, end_positions: np.ndarray, features: Dict[str, np.ndarray], labels: Dict[str, np.ndarray], seq_len: int):
        self.end_positions = end_positions.astype(np.int64)
        self.features = features
        self.labels = labels
        self.seq_len = seq_len

    def __len__(self):
        return len(self.end_positions)

    def __getitem__(self, idx):
        end = int(self.end_positions[idx])
        start = end - self.seq_len + 1
        sl = slice(start, end + 1)

        x = {
            "pc_id": torch.from_numpy(self.features["pc_id"][sl]).long(),
            "delta_id": torch.from_numpy(self.features["delta_id"][sl]).long(),
            "offset_id": torch.from_numpy(self.features["offset_id"][sl]).long(),
            "hit_id": torch.from_numpy(self.features["hit_id"][sl]).long(),
            "access_id": torch.from_numpy(self.features["access_id"][sl]).long(),
            "semantic_id": torch.from_numpy(self.features["semantic_id"][sl]).long(),
            "dt_bucket": torch.from_numpy(self.features["dt_bucket"][sl]).long(),
            "cont": torch.from_numpy(self.features["cont"][sl]).float(),
        }
        y = {
            "delta": torch.tensor(self.labels["delta"][end]).long(),
            "hit": torch.tensor(self.labels["hit"][end]).long(),
            "hit_mask": torch.tensor(self.labels["hit_mask"][end]).float(),
            "bypass": torch.tensor(self.labels["bypass"][end]).long(),
            "timing": torch.tensor(self.labels["timing"][end]).long(),
            "end_pos": torch.tensor(end).long(),
        }
        return x, y

train_ds = CacheSequenceDataset(train_end_positions, feature_arrays, label_arrays, seq_len)
val_ds = CacheSequenceDataset(val_end_positions, feature_arrays, label_arrays, seq_len)

train_loader = DataLoader(
    train_ds, batch_size=CONFIG["batch_size"], shuffle=True,
    num_workers=CONFIG["num_workers"], pin_memory=(DEVICE == "cuda")
)
val_loader = DataLoader(
    val_ds, batch_size=CONFIG["batch_size"], shuffle=False,
    num_workers=CONFIG["num_workers"], pin_memory=(DEVICE == "cuda")
)

batch = next(iter(train_loader))
{k: v.shape for k, v in batch[0].items()}


In [ ]:
# ============================================================
# 8. Model: multi-task LSTM cache action predictor
# ============================================================

class LSTMCacheActionPredictor(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        emb = cfg["emb_dim"]

        self.pc_emb = nn.Embedding(cfg["pc_hash_buckets"], emb)
        self.delta_emb = nn.Embedding(num_delta_classes, emb)
        self.offset_emb = nn.Embedding(cfg["offset_vocab"], emb // 2)
        self.hit_emb = nn.Embedding(3, emb // 4)
        self.access_emb = nn.Embedding(2, emb // 4)
        self.semantic_emb = nn.Embedding(cfg["semantic_hash_buckets"], emb // 2)
        self.dt_emb = nn.Embedding(num_time_classes, emb // 2)

        self.cont_proj = nn.Sequential(
            nn.Linear(len(cont_cols), cfg["cont_dim"]),
            nn.ReLU(),
        )

        in_dim = emb + emb + emb // 2 + emb // 4 + emb // 4 + emb // 2 + emb // 2 + cfg["cont_dim"]

        self.lstm = nn.LSTM(
            input_size=in_dim,
            hidden_size=cfg["hidden_dim"],
            num_layers=cfg["num_layers"],
            dropout=cfg["dropout"] if cfg["num_layers"] > 1 else 0.0,
            batch_first=True,
        )
        self.norm = nn.LayerNorm(cfg["hidden_dim"])
        self.trunk = nn.Sequential(
            nn.Linear(cfg["hidden_dim"], cfg["hidden_dim"]),
            nn.ReLU(),
            nn.Dropout(cfg["dropout"]),
        )

        self.delta_head = nn.Linear(cfg["hidden_dim"], num_delta_classes)
        self.hit_head = nn.Linear(cfg["hidden_dim"], 2)
        self.bypass_head = nn.Linear(cfg["hidden_dim"], 2)
        self.timing_head = nn.Linear(cfg["hidden_dim"], num_time_classes)

    def forward(self, x):
        parts = [
            self.pc_emb(x["pc_id"]),
            self.delta_emb(x["delta_id"].clamp(0, num_delta_classes - 1)),
            self.offset_emb(x["offset_id"].clamp(0, CONFIG["offset_vocab"] - 1)),
            self.hit_emb(x["hit_id"].clamp(0, 2)),
            self.access_emb(x["access_id"].clamp(0, 1)),
            self.semantic_emb(x["semantic_id"].clamp(0, CONFIG["semantic_hash_buckets"] - 1)),
            self.dt_emb(x["dt_bucket"].clamp(0, num_time_classes - 1)),
            self.cont_proj(x["cont"]),
        ]
        z = torch.cat(parts, dim=-1)
        out, _ = self.lstm(z)
        h = self.norm(out[:, -1, :])
        h = self.trunk(h)
        return {
            "delta": self.delta_head(h),
            "hit": self.hit_head(h),
            "bypass": self.bypass_head(h),
            "timing": self.timing_head(h),
        }

model = LSTMCacheActionPredictor(CONFIG).to(DEVICE)
print(model)
print("num parameters:", sum(p.numel() for p in model.parameters()))


In [ ]:
# ============================================================
# 9. Losses and metrics
# ============================================================

ce_delta = nn.CrossEntropyLoss()
ce_cls = nn.CrossEntropyLoss(reduction="none")

def to_device_batch(x, y, device):
    x = {k: v.to(device, non_blocking=True) for k, v in x.items()}
    y = {k: v.to(device, non_blocking=True) for k, v in y.items()}
    return x, y

def compute_loss(logits, y):
    loss_delta = ce_delta(logits["delta"], y["delta"])

    hit_raw = ce_cls(logits["hit"], y["hit"])
    loss_hit = (hit_raw * y["hit_mask"]).sum() / y["hit_mask"].sum().clamp_min(1.0)

    loss_bypass = ce_cls(logits["bypass"], y["bypass"]).mean()
    loss_timing = ce_delta(logits["timing"], y["timing"])

    total = (
        CONFIG["loss_delta"] * loss_delta
        + CONFIG["loss_hit"] * loss_hit
        + CONFIG["loss_bypass"] * loss_bypass
        + CONFIG["loss_timing"] * loss_timing
    )
    return total, {
        "delta": float(loss_delta.detach().cpu()),
        "hit": float(loss_hit.detach().cpu()),
        "bypass": float(loss_bypass.detach().cpu()),
        "timing": float(loss_timing.detach().cpu()),
    }

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total = 0
    loss_sum = 0.0
    delta_top1 = 0
    delta_top5 = 0
    hit_ok = 0
    hit_count = 0
    bypass_ok = 0
    timing_ok = 0

    # For bypass F1.
    tp = fp = fn = 0

    for x, y in loader:
        x, y = to_device_batch(x, y, DEVICE)
        logits = model(x)
        loss, _ = compute_loss(logits, y)
        bs = y["delta"].shape[0]
        total += bs
        loss_sum += float(loss.detach().cpu()) * bs

        pred_delta = logits["delta"].argmax(dim=-1)
        delta_top1 += (pred_delta == y["delta"]).sum().item()
        top5 = logits["delta"].topk(k=min(5, logits["delta"].shape[-1]), dim=-1).indices
        delta_top5 += (top5 == y["delta"].unsqueeze(1)).any(dim=1).sum().item()

        pred_hit = logits["hit"].argmax(dim=-1)
        hm = y["hit_mask"].bool()
        hit_ok += ((pred_hit == y["hit"]) & hm).sum().item()
        hit_count += hm.sum().item()

        pred_bypass = logits["bypass"].argmax(dim=-1)
        bypass_ok += (pred_bypass == y["bypass"]).sum().item()
        tp += ((pred_bypass == 1) & (y["bypass"] == 1)).sum().item()
        fp += ((pred_bypass == 1) & (y["bypass"] == 0)).sum().item()
        fn += ((pred_bypass == 0) & (y["bypass"] == 1)).sum().item()

        pred_timing = logits["timing"].argmax(dim=-1)
        timing_ok += (pred_timing == y["timing"]).sum().item()

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)

    return {
        "loss": loss_sum / max(total, 1),
        "delta_top1": delta_top1 / max(total, 1),
        "delta_top5": delta_top5 / max(total, 1),
        "future_hit_acc": hit_ok / max(hit_count, 1),
        "bypass_acc": bypass_ok / max(total, 1),
        "bypass_precision": precision,
        "bypass_recall": recall,
        "bypass_f1": f1,
        "timing_acc": timing_ok / max(total, 1),
    }


In [ ]:
# ============================================================
# 10. Training loop
# ============================================================

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(CONFIG["epochs"], 1))

best_score = -1.0
best_epoch = -1
bad_epochs = 0
history = []
ckpt_path = ARTIFACT_DIR / "lstm_cache_action_predictor.pt"

for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    t0 = time.time()
    total_loss = 0.0
    total_seen = 0

    for x, y in train_loader:
        x, y = to_device_batch(x, y, DEVICE)
        logits = model(x)
        loss, parts = compute_loss(logits, y)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
        optimizer.step()

        bs = y["delta"].shape[0]
        total_loss += float(loss.detach().cpu()) * bs
        total_seen += bs

    scheduler.step()
    metrics = evaluate(model, val_loader)

    # Balanced model-selection score. Not only delta accuracy.
    score = (
        0.45 * metrics["delta_top1"]
        + 0.20 * metrics["delta_top5"]
        + 0.20 * metrics["bypass_f1"]
        + 0.10 * metrics["future_hit_acc"]
        + 0.05 * metrics["timing_acc"]
    )

    row = {
        "epoch": epoch,
        "train_loss": total_loss / max(total_seen, 1),
        "score": score,
        "sec": time.time() - t0,
        **metrics,
    }
    history.append(row)
    print(row)

    if score > best_score:
        best_score = score
        best_epoch = epoch
        bad_epochs = 0
        torch.save({
            "model_state": model.state_dict(),
            "config": CONFIG,
            "vocab": vocab,
            "norm_stats": norm_stats,
            "cont_cols": cont_cols,
            "num_delta_classes": num_delta_classes,
            "num_time_classes": num_time_classes,
        }, ckpt_path)
        print("saved best checkpoint", ckpt_path)
    else:
        bad_epochs += 1
        if bad_epochs >= CONFIG["early_stop_patience"]:
            print("early stop at epoch", epoch)
            break

hist_df = pd.DataFrame(history)
hist_df.to_csv(ARTIFACT_DIR / "lstm_training_history.csv", index=False)
print("best_epoch", best_epoch, "best_score", best_score)
hist_df.tail()


In [ ]:
# ============================================================
# 11. Load best checkpoint and report accuracy gates
# ============================================================

ckpt = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.to(DEVICE)
val_metrics = evaluate(model, val_loader)
print(json.dumps(val_metrics, indent=2))

print("\nAccuracy gates / sanity checks:")
print("delta top1 >= target?", val_metrics["delta_top1"], ">=", CONFIG["target_delta_top1"])
print("delta top5 >= target?", val_metrics["delta_top5"], ">=", CONFIG["target_delta_top5"])
print("bypass f1 >= target?", val_metrics["bypass_f1"], ">=", CONFIG["target_bypass_f1"])

if val_metrics["delta_top1"] < CONFIG["target_delta_top1"]:
    print("[note] Delta top1 did not reach the paper-like gate. This may mean the trace is irregular or labels are too broad. Try PC-local training, larger vocab, or per-trace model.")
if val_metrics["bypass_f1"] < CONFIG["target_bypass_f1"]:
    print("[note] Bypass F1 did not reach target. Check reuse-distance threshold and whether the trace has enough repeated lines.")


In [ ]:
# ============================================================
# 12. Export action predictions for ChampSim replay
# ============================================================

@torch.no_grad()
def predict_loader(model, loader):
    model.eval()
    rows = []
    for x, y in loader:
        x, y = to_device_batch(x, y, DEVICE)
        logits = model(x)

        delta_prob = torch.softmax(logits["delta"], dim=-1)
        delta_conf, delta_id = delta_prob.max(dim=-1)

        bypass_prob = torch.softmax(logits["bypass"], dim=-1)[:, 1]
        hit_prob = torch.softmax(logits["hit"], dim=-1)[:, 1]
        timing_id = logits["timing"].argmax(dim=-1)

        for i in range(delta_id.shape[0]):
            did = int(delta_id[i].cpu())
            pred_delta = id_to_delta.get(did, 0)
            rows.append({
                "end_pos": int(y["end_pos"][i].cpu()),
                "pred_delta_id": did,
                "pred_delta": pred_delta,
                "pred_delta_conf": float(delta_conf[i].cpu()),
                "pred_future_hit_prob": float(hit_prob[i].cpu()),
                "pred_bypass_prob": float(bypass_prob[i].cpu()),
                "pred_timing_bin": int(timing_id[i].cpu()),
            })
    return pd.DataFrame(rows)

# Export validation predictions first; full export can be expensive but useful for replay.
val_pred = predict_loader(model, val_loader)

# Attach original event metadata.
meta_cols = ["trace", "event_id", "cycle_num", "pc_int", "addr_int", "line_addr"]
meta = df.iloc[val_pred["end_pos"].to_numpy()][meta_cols].reset_index(drop=True)
val_export = pd.concat([meta, val_pred.drop(columns=["end_pos"]).reset_index(drop=True)], axis=1)

def choose_action(row):
    # Direct action model:
    # - bypass is a cache-placement action
    # - prefetch delta is an address-generation/timing action
    if row["pred_bypass_prob"] >= CONFIG["prediction_threshold_bypass"]:
        return "BYPASS_OR_LOW_PRIORITY_INSERT"
    if row["pred_delta_conf"] >= CONFIG["prediction_threshold_prefetch"] and row["pred_delta"] != 0:
        return "PREFETCH_DELTA"
    return "INSERT_NORMAL_NO_PREFETCH"

val_export["nn_action"] = val_export.apply(choose_action, axis=1)
val_export["prefetch_line_addr"] = val_export["line_addr"] + val_export["pred_delta"]
val_export["prefetch_addr"] = (val_export["prefetch_line_addr"] * CONFIG["cache_line_bytes"]).astype(np.int64)

out_csv = ARTIFACT_DIR / "val_lstm_cache_actions.csv"
val_export.to_csv(out_csv, index=False)
print("saved", out_csv)
val_export.head(20)


In [ ]:
# ============================================================
# 13. Optional: full-trace prediction export
# ============================================================

if CONFIG["export_predictions"]:
    full_end_positions = np.where(valid_end)[0]
    full_ds = CacheSequenceDataset(full_end_positions, feature_arrays, label_arrays, CONFIG["seq_len"])
    full_loader = DataLoader(
        full_ds, batch_size=CONFIG["batch_size"], shuffle=False,
        num_workers=CONFIG["num_workers"], pin_memory=(DEVICE == "cuda")
    )
    full_pred = predict_loader(model, full_loader)
    meta = df.iloc[full_pred["end_pos"].to_numpy()][meta_cols].reset_index(drop=True)
    full_export = pd.concat([meta, full_pred.drop(columns=["end_pos"]).reset_index(drop=True)], axis=1)
    full_export["nn_action"] = full_export.apply(choose_action, axis=1)
    full_export["prefetch_line_addr"] = full_export["line_addr"] + full_export["pred_delta"]
    full_export["prefetch_addr"] = (full_export["prefetch_line_addr"] * CONFIG["cache_line_bytes"]).astype(np.int64)

    out_csv = ARTIFACT_DIR / "full_lstm_cache_actions.csv"
    full_export.to_csv(out_csv, index=False)
    print("saved", out_csv, "rows=", len(full_export))


## 14. How this will connect to ChampSim later

The file to consume from C++ / replay script is:

```text
formal_NN_training/artifacts/full_lstm_cache_actions.csv
```

Important columns:

```text
trace
event_id
cycle_num
pc_int
addr_int
line_addr
pred_delta
pred_delta_conf
pred_future_hit_prob
pred_bypass_prob
pred_timing_bin
nn_action
prefetch_line_addr
```

Suggested first real replay policy:

```text
if nn_action == "BYPASS_OR_LOW_PRIORITY_INSERT":
    insert with low priority or bypass selected cache level

elif nn_action == "PREFETCH_DELTA":
    prefetch prefetch_line_addr
    use pred_timing_bin to tune distance / degree

else:
    normal cache insertion, no NN prefetch
```

This makes the LSTM a **cache-action predictor**, not an SPP filter.

Replay helper scripts already live in:

```text
formal_NN_training/scripts/02_actions_to_prefetch_list.py
formal_NN_training/scripts/03_run_lstm_replay.sh
```


## 15. Replay exported LSTM actions through ChampSim

After the training/export cells have produced one of:

```text
formal_NN_training/artifacts/full_lstm_cache_actions.csv
formal_NN_training/artifacts/val_lstm_cache_actions.csv
```

this cell calls:

```text
formal_NN_training/scripts/03_run_lstm_replay.sh
```

The script converts the action table to a `list_replayer` prefetch list, then reuses the existing `projects/legacy_gru_prefetch/scripts/run_nn_replay.sh` flow.


In [ ]:
import os, subprocess
os.environ.setdefault('TRACE', '602.gcc_s-734B')
os.environ.setdefault('WARMUP', '25000000')
os.environ.setdefault('SIM', '25000000')
os.environ.setdefault('PREFETCH_THRESHOLD', str(CONFIG.get('prediction_threshold_prefetch', 0.50)))
os.environ.setdefault('BYPASS_THRESHOLD', str(CONFIG.get('prediction_threshold_bypass', 0.60)))

subprocess.run(['bash', 'formal_NN_training/scripts/03_run_lstm_replay.sh'], check=True)


## 16. Push generated data / artifacts / replay results

This stages only generated experiment outputs, not source edits. Use your authenticated remote from the Colab PAT setup before running `git push`.


In [ ]:
import subprocess, datetime
from pathlib import Path

paths_to_add = [
    'formal_NN_training/artifacts',
    'formal_NN_training/data/generated',
    'formal_NN_training/results',
    'results/generated/prefetch_lists',
    'results/nn_demo_summary.csv',
]

for p in paths_to_add:
    if Path(p).exists():
        subprocess.run(['git', 'add', p], check=True)

subprocess.run(['git', 'status', '--short'], check=True)

if subprocess.run(['git', 'diff', '--cached', '--quiet']).returncode != 0:
    msg = 'Add LSTM cache-action results ' + datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    subprocess.run(['git', 'commit', '-m', msg], check=True)
    subprocess.run(['git', 'push', 'origin', 'main'], check=True)
    print('pushed:', msg)
else:
    print('Nothing staged; no commit created.')
